# 03 Modelling

Reads what `python -m house_prices.train`, `evaluate` and `intervals` produced. Tuning and training happen in
those commands, not here, so the notebook stays quick and cannot touch the test set by accident. Two evaluation
setups are reported: a **random split** (shuffled 80/20, stratified by city) and a **time split** (train on older
listings, test on the newest 20%).

In [1]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from house_prices import clean, config, load

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid", context="notebook")
config.PATHS.figures_dir.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name):
    fig.savefig(config.PATHS.figures_dir / f"{name}.png", dpi=140, bbox_inches="tight")

In [2]:
import json

import joblib
from IPython.display import display

from house_prices import evaluate, features, formatting, split, train

metrics = json.loads(config.PATHS.metrics_json.read_text())
summaries = {s: json.loads((config.PATHS.candidates_dir / f"{s}_summary.json").read_text()) for s in split.SETUPS}
print("Synthetic data:", metrics["meta"]["synthetic_data"])
print("Deployed:", metrics["meta"]["deployed_model"], "trained with the", metrics["meta"]["deployed_setup"], "split")

Synthetic data: False
Deployed: xgboost trained with the random split


## The pipeline

Preprocessing sits inside one scikit-learn `Pipeline`, so cross-validation refits every step on each training
fold. The location target encoder is the important one: fitted on the full data it would leak the price of the
rows it later predicts.

In [3]:
from sklearn import set_config

set_config(display="text")
print(train.make_pipeline("lightgbm"))
print()
print("Inputs:", features.INPUT_COLUMNS)
print("Numeric:", features.NUMERIC_FEATURES)
print("Categorical (one-hot):", features.CATEGORICAL_FEATURES)
print(f"Location: grouped when fewer than {config.MIN_LOCATION_COUNT} training listings, then target-encoded (smoothed, 5-fold cross-fitted)")

Pipeline(steps=[('features', FeatureBuilder()), ('location', LocationGrouper()),
                ('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['area_sqft', 'log_area_sqft',
                                                   'bedrooms', 'baths',
                                                   'latitude', 'longitude',
                                                   'distance_to_centre_km']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['city', 'property_type']),
                                       

## Tuning setup

In [4]:
rows = []
for name in train.MODEL_NAMES:
    payload = summaries["random"]["summaries"][name]
    rows.append({"model": payload["label"], "search iterations": payload["n_iter"], "CV folds": payload["cv_folds"], "rows searched": payload["cv_rows"], "best parameters": payload["best_params"]})
display(pd.DataFrame(rows))
print("Search spaces (kind, low, high):")
for name in train.MODEL_NAMES:
    print(f"  {name}: {train.SEARCH_SPACES[name]}")

,model,search iterations,CV folds,rows searched,best parameters
0,Linear Regression,0,5,40000,{}
1,Ridge,8,5,40000,{'model__alpha': 8.471801418819979}
2,Random Forest,3,5,40000,"{'model__max_depth': 17, 'model__max_features'..."
3,XGBoost,8,5,40000,{'model__colsample_bytree': 0.6399899663272012...
4,LightGBM,8,5,40000,{'model__colsample_bytree': 0.7164916560792167...


Search spaces (kind, low, high):
  linear_regression: {}
  ridge: {'model__alpha': ('loguniform', 0.01, 100.0)}
  random_forest: {'model__n_estimators': ('randint', 120, 260), 'model__max_depth': ('randint', 10, 24), 'model__min_samples_leaf': ('randint', 1, 8), 'model__max_features': ('uniform', 0.4, 0.6)}
  xgboost: {'model__n_estimators': ('randint', 300, 800), 'model__learning_rate': ('loguniform', 0.03, 0.2), 'model__max_depth': ('randint', 4, 10), 'model__subsample': ('uniform', 0.7, 0.3), 'model__colsample_bytree': ('uniform', 0.6, 0.4), 'model__min_child_weight': ('randint', 1, 10), 'model__reg_lambda': ('loguniform', 0.5, 10.0)}
  lightgbm: {'model__n_estimators': ('randint', 300, 900), 'model__learning_rate': ('loguniform', 0.03, 0.2), 'model__num_leaves': ('randint', 31, 200), 'model__min_child_samples': ('randint', 10, 60), 'model__subsample': ('uniform', 0.7, 0.3), 'model__colsample_bytree': ('uniform', 0.6, 0.4), 'model__reg_lambda': ('loguniform', 0.5, 10.0)}


## Cross-validation on the training set only

In [5]:
def cv_table(setup):
    rows = []
    for name in evaluate.MODEL_ORDER:
        cv = summaries[setup]["summaries"][name]["cv"]
        rows.append(
            {
                "model": evaluate.MODEL_LABELS[name],
                "RMSE (log) mean": cv["rmse_log"]["mean"],
                "RMSE (log) std": cv["rmse_log"]["std"],
                "R2 (log) mean": cv["r2_log"]["mean"],
                "MdAPE mean": cv["mdape"]["mean"],
                "MAE (PKR) mean": cv["mae_pkr"]["mean"],
            }
        )
    return pd.DataFrame(rows).round(4)


for setup in split.SETUPS:
    print(f"CV on the {setup} split's training set")
    display(cv_table(setup))

CV on the random split's training set


,model,RMSE (log) mean,RMSE (log) std,R2 (log) mean,MdAPE mean,MAE (PKR) mean
0,Baseline: overall median price,0.9963,0.0084,-0.0043,0.6064,1.767027e+07
1,Baseline: location median price/sq ft x size,0.4228,0.0049,0.8191,0.2200,7.523186e+06
2,Linear Regression,0.3799,0.0050,0.8539,0.2159,6.975756e+06
3,Ridge,0.3791,0.0045,0.8545,0.2160,7.008411e+06
4,Random Forest,0.3014,0.0035,0.9081,0.1508,5.291461e+06
5,XGBoost,0.2765,0.0036,0.9226,0.1408,4.868174e+06
6,LightGBM,0.2767,0.0032,0.9225,0.1419,4.961181e+06


CV on the time split's training set


,model,RMSE (log) mean,RMSE (log) std,R2 (log) mean,MdAPE mean,MAE (PKR) mean
0,Baseline: overall median price,0.9888,0.0044,-0.0051,0.6171,1.781687e+07
1,Baseline: location median price/sq ft x size,0.4046,0.0029,0.8317,0.2171,7.670149e+06
2,Linear Regression,0.3673,0.0020,0.8613,0.2142,6.985063e+06
3,Ridge,0.3673,0.0020,0.8613,0.2143,6.988142e+06
4,Random Forest,0.2949,0.0032,0.9106,0.1496,5.306154e+06
5,XGBoost,0.2666,0.0041,0.9269,0.1361,4.874751e+06
6,LightGBM,0.2663,0.0040,0.9270,0.1361,4.866116e+06


## Location encoding: target encoding against one-hot

LightGBM with the tuned parameters, cross-validated on the training set of the deployed split.

In [6]:
ablation = metrics.get("encoding_ablation")
if ablation:
    display(pd.DataFrame({k: {m: v[m]["mean"] for m in ("rmse_log", "r2_log", "mdape", "mae_pkr")} for k, v in ablation.items()}).T.round(4))

,rmse_log,r2_log,mdape,mae_pkr
target_encoding,0.2767,0.9225,0.1419,4.961181e+06
one_hot,0.2762,0.9228,0.1415,4.965039e+06


## Test-set results, both setups

The test rows were used for the first time in `python -m house_prices.evaluate`.

In [7]:
def test_table(setup):
    rows = []
    for name in evaluate.MODEL_ORDER:
        t = metrics["setups"][setup]["models"][name]["test"]
        rows.append(
            {
                "model": evaluate.MODEL_LABELS[name],
                "MAE": formatting.format_pkr(t["mae_pkr"]),
                "RMSE": formatting.format_pkr(t["rmse_pkr"]),
                "MdAPE": f"{t['mdape']:.1%}",
                "MAPE": f"{t['mape']:.1%}",
                "R2 (log)": round(t["r2_log"], 3),
            }
        )
    return pd.DataFrame(rows)


for setup in split.SETUPS:
    s = metrics["setups"][setup]
    print(f"{setup} split: train {s['train_rows']:,} ({s['train_date_range'][0]} to {s['train_date_range'][1]}), test {s['test_rows']:,} ({s['test_date_range'][0]} to {s['test_date_range'][1]}); chosen model {evaluate.MODEL_LABELS[s['chosen_model']]}")
    display(test_table(setup))

random split: train 53,658 (2018-08-05 to 2019-08-06), test 13,415 (2018-08-05 to 2019-08-06); chosen model XGBoost


,model,MAE,RMSE,MdAPE,MAPE,R2 (log)
0,Baseline: overall median price,PKR 1.76 Crore,PKR 3.96 Crore,61.4%,96.8%,-0.006
1,Baseline: location median price/sq ft x size,PKR 75.9 Lakh,PKR 3.03 Crore,21.8%,35.2%,0.820
2,Linear Regression,PKR 68.16 Lakh,PKR 1.85 Crore,21.1%,30.4%,0.857
3,Ridge,PKR 68.44 Lakh,PKR 1.86 Crore,21.1%,30.4%,0.857
4,Random Forest,PKR 49.75 Lakh,PKR 1.56 Crore,14.5%,21.5%,0.915
5,XGBoost,PKR 47.31 Lakh,PKR 1.45 Crore,13.6%,19.7%,0.929
6,LightGBM,PKR 47.02 Lakh,PKR 1.51 Crore,13.7%,19.8%,0.928


time split: train 53,524 (2018-08-05 to 2019-07-03), test 13,549 (2019-07-04 to 2019-08-06); chosen model LightGBM


,model,MAE,RMSE,MdAPE,MAPE,R2 (log)
0,Baseline: overall median price,PKR 1.63 Crore,PKR 3.55 Crore,62.7%,118.1%,-0.002
1,Baseline: location median price/sq ft x size,PKR 73.47 Lakh,PKR 2.45 Crore,22.8%,44.2%,0.772
2,Linear Regression,PKR 64.92 Lakh,PKR 1.57 Crore,22.0%,37.5%,0.821
3,Ridge,PKR 64.94 Lakh,PKR 1.57 Crore,22.0%,37.5%,0.821
4,Random Forest,PKR 49.1 Lakh,PKR 1.31 Crore,15.4%,26.6%,0.888
5,XGBoost,PKR 47.92 Lakh,PKR 1.39 Crore,14.5%,23.1%,0.910
6,LightGBM,PKR 46.89 Lakh,PKR 1.28 Crore,14.6%,23.2%,0.910


In [8]:
fig = plt.figure(figsize=(13, 4.8))
from PIL import Image

image = Image.open(config.PATHS.figures_dir / "model_comparison.png")
plt.imshow(image)
plt.axis("off")
plt.show()

## Random split against time split

In [9]:
lines = []
for name in evaluate.MODEL_ORDER:
    r = metrics["setups"]["random"]["models"][name]["test"]
    t = metrics["setups"]["time"]["models"][name]["test"]
    lines.append(f"{evaluate.MODEL_LABELS[name]}: median error {r['mdape']:.1%} on the random split, {t['mdape']:.1%} on the time split ({t['mdape'] - r['mdape']:+.1%} points).")
print("\n".join("- " + l for l in lines))
chosen_r = metrics["setups"]["random"]["chosen_model"]
chosen_t = metrics["setups"]["time"]["chosen_model"]
cv_r = metrics["setups"]["random"]["models"][chosen_r]["cv"]["rmse_log"]["mean"]
te_r = metrics["setups"]["random"]["models"][chosen_r]["test"]["rmse_log"]
te_t = metrics["setups"]["time"]["models"][chosen_t]["test"]["rmse_log"]
print()
print(f"Chosen model, RMSE on log price: CV {cv_r:.3f}, random-split test {te_r:.3f}, time-split test {te_t:.3f}.")

- Baseline: overall median price: median error 61.4% on the random split, 62.7% on the time split (+1.2% points).
- Baseline: location median price/sq ft x size: median error 21.8% on the random split, 22.8% on the time split (+1.0% points).
- Linear Regression: median error 21.1% on the random split, 22.0% on the time split (+1.0% points).
- Ridge: median error 21.1% on the random split, 22.0% on the time split (+0.9% points).
- Random Forest: median error 14.5% on the random split, 15.4% on the time split (+0.9% points).
- XGBoost: median error 13.6% on the random split, 14.5% on the time split (+0.9% points).
- LightGBM: median error 13.7% on the random split, 14.6% on the time split (+1.0% points).

Chosen model, RMSE on log price: CV 0.276, random-split test 0.267, time-split test 0.304.


A shuffled split lets the model see listings from the same weeks as its test rows (the same agents, near
duplicates, the same market level). A time split asks it to price listings from a later period, which is how it
would be used. When the time-split error is larger, that gap is the honest number to quote.

## 80% prediction interval

In [10]:
rows = []
for setup, block in metrics["intervals"].items():
    o = block["overall"]
    rows.append({"split": setup, "coverage": f"{o['coverage']:.1%}", "target": f"{block['level']:.0%}", "mean width": formatting.format_pkr(o["mean_width_pkr"]), "mean width / estimate": f"{o['mean_relative_width']:.0%}"})
display(pd.DataFrame(rows))
for setup, block in metrics["intervals"].items():
    print(f"Coverage by city, {setup} split")
    display(pd.DataFrame(block["by_city"]).assign(coverage=lambda d: (d["coverage"] * 100).round(1), mean_width_pkr=lambda d: d["mean_width_pkr"].round(0)))

,split,coverage,target,mean width,mean width / estimate
0,random,73.8%,80%,PKR 1.29 Crore,54%
1,time,67.8%,80%,PKR 1.12 Crore,53%


Coverage by city, random split


,city,n,coverage,mean_width_pkr,median_width_pkr,mean_relative_width
0,Faisalabad,374,70.1,8204804.0,4.965103e+06,0.611483
1,Islamabad,2069,71.8,15532019.0,7.075803e+06,0.501327
2,Karachi,5008,75.3,13709312.0,6.493339e+06,0.594115
3,Lahore,4624,73.2,12725449.0,6.632205e+06,0.484212
4,Rawalpindi,1340,74.5,8021616.0,5.356318e+06,0.544698


Coverage by city, time split


,city,n,coverage,mean_width_pkr,median_width_pkr,mean_relative_width
0,Faisalabad,538,65.8,6738663.0,4.410281e+06,0.528179
1,Islamabad,2391,67.2,12756547.0,6.559642e+06,0.448739
2,Karachi,5188,67.0,11808139.0,6.019674e+06,0.622847
3,Lahore,3741,69.0,12120596.0,5.947007e+06,0.452934
4,Rawalpindi,1691,69.3,6797639.0,4.941882e+06,0.509981
